## ML_1036: LoRA (Low-Rank Adaptation)

**Difficulty**: Medium | **TorchCode**: #26

### Problem
Implement LoRA: freeze a base `nn.Linear` and add two low-rank adapter matrices A and B. Only A and B are trained. B is initialized to zero so the adapter starts as an identity perturbation.

### Formula
$$y = W_0 x + \frac{\alpha}{r} B A x, \quad A \in \mathbb{R}^{r \times d_{\text{in}}}, \; B \in \mathbb{R}^{d_{\text{out}} \times r}, \; B_0 = 0$$

In [ ]:
import torch
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad_(False)
        self.linear.bias.requires_grad_(False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        self.scaling = alpha / rank

    def forward(self, x):
        return self.linear(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

In [ ]:
import torch
import torch.nn as nn

# --- Test 1: Base weights frozen ---
layer = LoRALinear(in_features=16, out_features=8, rank=4)
assert isinstance(layer, nn.Module)
assert not layer.linear.weight.requires_grad
assert not layer.linear.bias.requires_grad

# --- Test 2: LoRA parameter shapes ---
assert layer.lora_A.shape == (4, 16)
assert layer.lora_B.shape == (8, 4)

# --- Test 3: B=0 means output equals base ---
torch.manual_seed(0)
layer2 = LoRALinear(in_features=8, out_features=4, rank=2)
x = torch.randn(2, 8)
assert torch.allclose(layer2(x), layer2.linear(x), atol=1e-5)

# --- Test 4: Only LoRA params get gradients ---
layer2(x).sum().backward()
assert layer2.lora_A.grad is not None
assert layer2.lora_B.grad is not None
assert layer2.linear.weight.grad is None

# --- Test 5: Forward computation ---
torch.manual_seed(0)
layer3 = LoRALinear(in_features=8, out_features=4, rank=2, alpha=2.0)
layer3.lora_B.data.normal_()
x = torch.randn(3, 8)
ref = layer3.linear(x) + (x @ layer3.lora_A.T @ layer3.lora_B.T) * (2.0 / 2)
assert torch.allclose(layer3(x), ref, atol=1e-5)

print('All tests passed!')